# 多步推理与规划


> 前面几讲分别缓解了单步推理的局限：第 2 讲用重复采样把计算花在推理时，第 3 讲用验证器给一条解打分，第 4 讲让模型与环境交替、拿到工具反馈。但这些方案都在同一条路径上推进——如果这一步选错了，后面无法回头。
>
> 这一节把多步任务当作规划问题处理：该拆成几步、该保留几条候选、哪些步骤能并行、每一步该加宽还是加深。我们用 ADaPT、LATS、SPRINT、Wider or Deeper 四篇论文的思路分别回答，并用 numpy 从零实现它们的核心循环。


模型每次调用只能输出一步动作，可现实里的任务几乎都要很多步才能完成。比如让模型写一份课程笔记：先要确定主题、搜集资料、列出提纲，再逐节写作、最后校对，前后步骤有依赖，顺序不能乱。如果模型不先把这些步骤想清楚，直接动手，往往写一半就发现结构乱了。把"一个任务按什么顺序做哪些动作"这件事预先定下来，就是规划。

第 4 讲的 ReAct 循环已经能让模型一步步行动，但它每一步只沿着当前最好的方向走，中间任何一步选错，后面就很难回头。多步任务真正难的往往不在单步动作本身，而在动手之前：要不要把任务拆开、拆成几块、先做哪块。这些问题想不清楚，走再多步也可能在错误的方向上越走越远。

规划最常见的一种方式，是把一个大任务切成几个小任务，逐个完成。比如制作一把木剑，要先得到木板和铁锭，把这两个子目标分别搞定，木剑自然就出来了。这种把任务拆成子目标、再按顺序完成的思路，叫任务分解。

还有一种情况：通往目标的路线不止一条，先试哪条都可能错。这时可以同时保留几条候选路线，一条走不通就换另一条。这种同时尝试多条路线的思路，叫树搜索。

学完这一讲，我们能自己实现一个"先规划再做"的 Agent 循环：模型先决定动作的整体结构，再一步步执行。这一讲顺着四个问题展开：怎么把任务拆开、怎么同时保留多条候选、哪些步骤能并行、每一步该多试几条路还是往深想一层。放进第 1 讲的感知、决策、行动、反馈循环里看，这一讲强化的是决策这一步——从每一步只选一个动作，升级成先规划再行动。先从单步推理卡在哪里说起。

这一节先用一个小例子看清楚单步推理卡在哪里，后面的分解与搜索都是针对这个问题提出的解法。

让模型处理多步任务，最朴素的做法有两种：让模型一次输出完整答案（整体生成），或者一步步往前走（贪心解码）。整体生成在长任务上不可靠，中间任何一步错了，结尾就会跟着错。贪心解码每一步只保留当前最好的选择，一旦选错就回不到更早的状态。

用一个数字运算链来说明。从 1 出发，依次做三次运算，目标值是 40。正确步骤是 ×4、+6、×4。如果模型在第 2 步输出 +2，这条路径的终值就变成 24；之后继续做正确步骤，也无法把 24 拉回 40——错值已经参与了后续计算，而单条路径没有保存其他可能。

In [ ]:
# 一条单步贪心路径：从 1 出发依次做 3 个运算，目标值 40。
# 正确步骤是 [×4, +6, ×4]，第 2 步会被替换成 +2。
true_steps = [("×", 4), ("+", 6), ("×", 4)]
bad_steps = [("×", 4), ("+", 2), ("×", 4)]

def run_chain(steps):
    """按给定运算链从 1 出发，返回终值。"""
    value = 1
    for op, x in steps:
        value = value * x if op == "×" else value + x
    return value

print("贪心单路径（第 2 步换成 +2）的终值:", run_chain(bad_steps))
print("目标值:", 40)

# 维护多条候选：每一步同时保留两条走法，错误分支不会挤掉正确分支
frontier = [[1]]
for i in range(3):
    nxt = []
    for path in frontier:
        for steps in (true_steps, bad_steps):
            op, x = steps[i]
            cur = path[-1]
            nxt.append(path + [cur * x if op == "×" else cur + x])
    frontier = nxt

values = sorted(p[-1] for p in frontier)
print("保留两条候选后，各路径终值:", values)
print("到达 40 的路径数:", sum(1 for v in values if v == 40))


上面两行输出对比两种做法。单条路径只保存一条轨迹：第 2 步用了 +2，终值停在 24，后面再按正确步骤 ×4 也回不到 40。

保留两条候选的做法，每一步都对每条旧路径按两种策略各扩展一次，路径数从 1 增长到 2、4、8。第 1 步和第 3 步两种策略恰好都是 ×4，真正不同的只有第 2 步的 +6 与 +2，所以 8 条路径只落回两个终值：走 +6 的 4 条到 40，走 +2 的 4 条停在 24。排序输出 [24, 24, 24, 24, 40, 40, 40, 40] 正是这个结果。

关键在于保留：即使第 2 步先看到 +2 这条分支，+6 的分支也没有被丢弃，正确的终值 40 仍然存在。树搜索要做的事情，就是把这种保留扩展到每一步、每一个状态。

上一节的例子说明，一条路走到底会卡住。这一节讲第一种解法：把任务拆成小块。关键问题是什么时候该拆、拆多细，ADaPT 给出的答案是：需要时才拆。

分解是把大任务切成小任务。最直接的做法是先列完整计划再执行（plan-and-execute）：一开始就把任务拆到最小步骤，然后按顺序执行。它的局限在于无法预知哪个子任务难、哪个简单——对能一步完成的子任务也强行拆细，反而引入多余动作和错误假设。

ADaPT 换了一个顺序：先让执行器（executor）直接尝试整个任务；执行器自己报告失败时，才让规划器（planner）把任务拆成若干子任务，再对每个子任务递归调用同一套流程。拆的深度由任务的真实难度决定，而不是事先固定。执行器把"我完成了"或"我失败了"作为输出，这个自报结果充当任务的成功信号。

两个角色分开：planner 负责把任务拆成子任务，executor 负责实际执行，递归控制由一个固定程序 controller 完成。子任务之间有两种组合方式：AND 表示必须全部成功，OR 表示任一成功即可。

这一小节搭一个最小环境，用来比较三种分解策略，目的是看清"需要时才拆"到底省了什么。

每种物品有一个配方（需要的子物品），原子物品不需要任何子物品。executor 能直接完成原子物品和一步可做的物品（配方里的子物品全部是原子的）；需要更深组合的物品，executor 会失败。这样一个"executor 能力有限"的环境，能凸显按需分解的必要性。

In [ ]:
# 合成配方环境：物品 → 需要的子物品（空列表表示原子物品）
RECIPES = {
    "木棍": [],
    "铁锭": [],
    "木板": ["木棍"],
    "木剑": ["木板", "铁锭"],
    "武器箱": ["木剑", "铁锭"],
}

def is_atomic(item):
    """物品是否原子：不需要任何子物品。"""
    return len(RECIPES[item]) == 0

def is_direct(item):
    """物品是否一步可做：配方里的子物品全部原子。"""
    return all(is_atomic(sub) for sub in RECIPES[item])

def executor_ability(item):
    """executor 的完成度判定：原子或一步可做的物品能直接完成，否则失败。"""
    if is_atomic(item) or is_direct(item):
        return "completed"
    return "failed"

for item in RECIPES:
    print(f"{item}: 配方={RECIPES[item]}, executor={executor_ability(item)}")


executor 的判定只有两行：物品是原子，或配方里的子物品全部是原子，就判定 completed，否则 failed。逐项核对：

| 物品 | 配方 | 子物品是否全部原子 | executor 判定 |
|:---|:---|:---|:---|
| 木棍 | [] | 是（无需子物品） | completed |
| 铁锭 | [] | 是（无需子物品） | completed |
| 木板 | [木棍] | 木棍是原子，是 | completed |
| 木剑 | [木板, 铁锭] | 木板不是原子，否 | failed |
| 武器箱 | [木剑, 铁锭] | 木剑不是原子，否 | failed |

executor 完成不了木剑，不是因为配方写错，而是因为木剑依赖的木板还有一层依赖（木板→木棍），组合深度超过一步。executor 被刻意设成只能做一步，这样一个弱执行器才能让分解的必要性显现出来。如果 executor 什么都能做，就不需要规划了。

In [ ]:
# 定位仓库根目录的 llm_client.py，统一创建客户端
import sys, os
_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, 'llm_client.py')):
    _root = os.path.dirname(_root)
    if _root == os.path.dirname(_root):
        break
if _root not in sys.path:
    sys.path.insert(0, _root)
from llm_client import get_llm

client = get_llm()
print("当前客户端是否为 真实 API 演示:", False)

def planner(client, item):
    """规划器：把任务拆成若干子任务，返回 (子任务列表, 组合逻辑)。
    真实模式询问 LLM 并宽容解析；真实 API 演示返回配方作为脚本化计划。"""
    if True:
        prompt = f"把「制作 {item}」拆成若干子任务，用 AND 连接，输出形如 A AND B 的计划。"
        reply = client.chat([{"role": "user", "content": prompt}])
        subs = [s.strip() for s in reply.replace("AND", " and ").split(" and ")]
        subs = [s for s in subs if s in RECIPES]
        if subs:
            return subs, "AND"
    return list(RECIPES[item]), "AND"

def executor(client, item, depth):
    """执行器：返回 (完成度, 自评文本)。
    完成度用环境的确定性规则判定，自评文本来自 LLM（脚本化示例 为脚本化占位）。"""
    outcome = executor_ability(item)
    if False:
        report = f"脚本化示例 占位：执行「{item}」→ {outcome}"
    else:
        msg = f"请执行「制作 {item}」并自评，完成输出 completed，失败输出 failed。"
        report = client.chat([{"role": "user", "content": msg}])
    return outcome, report

print("planner 输出:", planner(client, "木剑"))
print("executor 输出:", executor(client, "木剑", 0))


In [ ]:
# ADaPT 递归控制器：先试 executor，失败才让 planner 拆，拆完递归
def adapt_controller(client, item, depth, dmax, trace, path):
    """ADaPT(Task, depth)：depth 超过 dmax 只跑 executor。
    trace 记录调用树，(path, item, action, detail)，path 是祖先序列。"""
    outcome, _ = executor(client, item, depth)
    trace.append((path, item, "executor", outcome))
    if outcome == "completed":
        return True
    if depth >= dmax:
        return False
    subs, logic = planner(client, item)
    trace.append((path, item, "split", list(subs)))
    results = [adapt_controller(client, s, depth + 1, dmax, trace, path + (item,))
               for s in subs]
    return all(results) if logic == "AND" else any(results)

def run_react(client, item):
    """ReAct 基线：只跑一次 executor，不做任何分解。"""
    trace = []
    outcome, _ = executor(client, item, 0)
    trace.append(((), item, "executor", outcome))
    return trace, outcome == "completed"

def run_plan_execute(client, item):
    """Plan-and-Execute 基线：一次性拆到原子物品，再逐个执行。"""
    trace = []
    def plan_all(it, path):
        if is_atomic(it):
            outcome, _ = executor(client, it, len(path))
            trace.append((path, it, "executor", outcome))
            return
        trace.append((path, it, "split", list(RECIPES[it])))
        for sub in RECIPES[it]:
            plan_all(sub, path + (it,))
    plan_all(item, ())
    return trace, True


In [ ]:
dmax = 3
item = "武器箱"

react_trace, react_ok = run_react(client, item)
plan_trace, plan_ok = run_plan_execute(client, item)
adapt_trace = []
adapt_ok = adapt_controller(client, item, 0, dmax, adapt_trace, ())

def show_trace(trace):
    """把调用树打印成文本。"""
    lines = []
    for path, it, action, detail in trace:
        indent = "  " * len(path)
        if action == "split":
            lines.append(f"{indent}{it}: 拆成 {detail}")
        else:
            lines.append(f"{indent}{it}: {detail}")
    return "\n".join(lines)

print("=== ReAct（不分解）===")
print(show_trace(react_trace))
print("整体成功:", react_ok)
print()

print("=== Plan-and-Execute（一次性全拆）===")
print(show_trace(plan_trace))
print("整体成功:", plan_ok)
print()

print("=== ADaPT（失败才拆）===")
print(show_trace(adapt_trace))
print("整体成功:", adapt_ok)

def count_actions(trace):
    """统计 executor 与 split 的出现次数。"""
    n_exec = sum(1 for _, _, a, _ in trace if a == "executor")
    n_split = sum(1 for _, _, a, _ in trace if a == "split")
    return n_exec, n_split

for name, trace in [("ReAct", react_trace), ("Plan-and-Execute", plan_trace),
                    ("ADaPT", adapt_trace)]:
    ne, ns = count_actions(trace)
    print(f"{name}: executor 调用 {ne} 次, 分解 {ns} 次")

print("解读：ADaPT 只在失败处分解（武器箱、木剑），木板由 executor 直接完成；"
      "Plan-and-Execute 把木板也拆成了木棍。少一次分解，多几次失败的 executor 尝试。")


对照输出里的 ADaPT 一栏，看武器箱的递归过程：

1. executor 直接试武器箱，判定 failed——武器箱需要木剑，木剑又需要木板，executor 只能做一步。
2. planner 把武器箱拆成 [木剑, 铁锭]，用 AND 连接，意思是两样都必须成功。
3. 递归处理木剑：executor 再试，仍然 failed；planner 拆成 [木板, 铁锭]。
4. 递归处理木板：executor 一次完成（子物品木棍是原子）；铁锭同样一次完成。
5. 木剑的两个子任务都成功，AND 成立，木剑成功；铁锭也成功，武器箱成功。

整个 trace 只有武器箱、木剑两处分解，executor 调用 5 次、分解 2 次。对照 Plan-and-Execute：它先把木板也拆成 [木棍]，多一次多余的分解，而 executor 本来就能直接完成木板。ReAct 不做分解，武器箱直接失败。

按需体现在第 4 步：木板没有被强制拆细，因为 executor 已经能完成。拆的深度由执行结果决定，而不是预先写死。dmax=3 是一道保险，防止递归无限深入。

In [ ]:
import matplotlib.pyplot as plt

# 物品的英文标签，图上文字统一用英文
EN = {"木棍": "stick", "铁锭": "iron", "木板": "plank",
      "木剑": "sword", "武器箱": "armory"}

def build_tree(trace):
    """把 trace 还原成节点信息与父子关系。"""
    info, children, roots = {}, {}, []
    for path, it, action, detail in trace:
        nid = tuple(list(path) + [it])
        info.setdefault(nid, (it, action, detail))
        children.setdefault(nid, [])
        if path:
            children.setdefault(path, []).append(nid)
        else:
            roots.append(nid)
    return info, children, roots

def layout(nid, children, counter):
    """自底向上布置坐标：叶子依次编号，内部节点取子节点中点。"""
    if not children.get(nid):
        x = counter[0]
        counter[0] += 1
        return {nid: x}, x, x
    result, xmin, xmax = {}, 1e9, -1e9
    for c in children.get(nid, []):
        sub, lo, hi = layout(c, children, counter)
        result.update(sub)
        xmin, xmax = min(xmin, lo), max(xmax, hi)
    result[nid] = (xmin + xmax) / 2
    return result, xmin, xmax

def plot_trace(trace, title, ax):
    """画调用树：split 节点画方框，executor 节点按完成/失败着色。"""
    info, children, roots = build_tree(trace)
    pos, counter = {}, [0]
    for root in roots:
        sub, _, _ = layout(root, children, counter)
        pos.update(sub)
    depth = {nid: len(nid) - 1 for nid in info}
    for nid in info:
        for c in children.get(nid, []):
            ax.plot([pos[nid], pos[c]], [-depth[nid], -depth[c]],
                    color="#90a4ae", lw=1, zorder=1)
    for nid, (item, action, detail) in info.items():
        x, y = pos[nid], -depth[nid]
        en = EN.get(item, item)
        if action == "executor":
            color = "#4caf50" if detail == "completed" else "#ef5350"
            ax.scatter(x, y, s=2200, c=color, zorder=3)
            ax.text(x, y, en, ha="center", va="center", fontsize=9, zorder=4)
        else:
            ax.add_patch(plt.Rectangle((x - 0.25, y - 0.18), 0.5, 0.36,
                                       fill=False, edgecolor="#455a64", lw=1.5))
            ax.text(x, y, en, ha="center", va="center", fontsize=9)
    ax.set_xticks([])
    ax.set_ylim(-max(depth.values()) - 0.6, 0.5)
    ax.set_ylabel("depth")
    ax.set_title(title)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
plot_trace(react_trace, "ReAct", axes[0])
plot_trace(plan_trace, "Plan-and-Execute", axes[1])
plot_trace(adapt_trace, "ADaPT", axes[2])
plt.tight_layout()
plt.show()


这一节回答：一条路线走错时，能不能同时保留多条候选一起试。树搜索就是这个思路。

分解把任务在垂直方向切细，降低每一步的难度；树搜索在水平方向展开——同一个状态保留多个候选动作，走错一条还能回到其他分支。蒙特卡洛树搜索（MCTS）把搜索组织成一棵树：节点是状态，边是一个候选动作，根是初始状态，叶子是尚未展开的状态。

LATS 让同一个 LLM 扮演三个角色：Agent 采样候选动作，价值函数评估状态，反思器从失败中总结教训。它成立的前提是 LLM 任务可回退——想回到任何历史状态，把此前的文本重新作为输入即可，不需要模拟世界。这一节先手算两个核心公式（UCT 选择与价值回溯），再在 24 点环境上跑一个简化 LATS。

这一小节把搜索里的两个环节各手算一遍：选哪个子节点往下走（选择），以及把结果带回给祖先节点（回溯）。先看选择。

选择阶段用 UCT 公式在子节点中挑一个：

$$ UCT(s) = V(s) + w\sqrt{\frac{\ln N(p)}{N(s)}} $$

V(s) 是子节点的价值，N(s) 是子节点访问次数，N(p) 是父节点访问次数，w 控制探索强度。第一项偏向访问过的、价值高的节点（利用），第二项偏向访问次数少的节点（探索）：N(s) 小的时候 $\ln N(p)/N(s)$ 大，未充分探索的子节点会被优先尝试。对数是放在分子里的，因为父节点访问次数增长时，探索的需求只缓慢增加。

用一棵手工构造的小树验证。根节点访问 24 次，四个子节点各有价值和访问次数：

| 子节点 | V(s) | N(s) |
|:---|:---|:---|
| s1 | 0.30 | 10 |
| s2 | 0.50 | 5 |
| s3 | 0.10 | 8 |
| s4 | 0.00 | 1 |

w=1 时，s4 的探索项约为 1.78，虽然价值为 0，总分仍然最高，会被优先展开。下面改变 w 观察选择如何切换。

把四个子节点逐个代入公式，验证 s4 为什么被选中。先算分子的 ln(24)≈3.178，再算每个子节点的探索项 sqrt(3.178 / N(s))：

| 子节点 | V(s) | N(s) | 探索项 sqrt(3.178/N) | UCT 总分 |
|:---|:---|:---|:---|:---|
| s1 | 0.30 | 10 | sqrt(0.318) ≈ 0.564 | 0.864 |
| s2 | 0.50 | 5 | sqrt(0.636) ≈ 0.797 | 1.297 |
| s3 | 0.10 | 8 | sqrt(0.397) ≈ 0.630 | 0.730 |
| s4 | 0.00 | 1 | sqrt(3.178) ≈ 1.783 | 1.783 |

s4 的价值是 0，访问次数只有 1，但探索项 1.783 是四者中最大的，总分 1.783 反超 s2 的 1.297。它访问最少，我们对它的了解最少，值得先试一次。

再注意 ln 的位置。分子是父节点的访问次数：ln(24)≈3.18，ln(1000)≈6.9，父节点访问次数从 24 涨到 1000，对数只涨了一倍多。探索需求不会随父访问次数直线上升，这正是设计上用对数的原因。

In [ ]:
import numpy as np

children = {"s1": (0.30, 10), "s2": (0.50, 5), "s3": (0.10, 8), "s4": (0.00, 1)}
N_parent = 24.0

def uct(v, n, w):
    """UCT 分数：价值 + 探索项。"""
    return v + w * np.sqrt(np.log(N_parent) / n)

for w in [0.1, 1.0, 3.0]:
    scores = {k: uct(*val, w) for k, val in children.items()}
    best = max(scores, key=scores.get)
    fmt = "  ".join(f"{k}={s:.3f}" for k, s in scores.items())
    print(f"w={w}: {fmt}  -> 选 {best}")


三个 w 的结果正好演示利用与探索的切换。

w=0.1 时，探索项被压到 0.06–0.18，几乎不起作用，总分由价值主导，s2（价值 0.50 最高）胜出，这是利用。
w=1.0 时，探索项回到 0.56–1.78，s4 的探索项最大，反超价值更高的 s2，这是探索。
w=3.0 时，探索项再放大三倍，s4 以 5.348 压倒性胜出。

w 是探索的权重：越小越偏向价值高的节点，越大越优先碰没访问过的节点。w 是需要调节的超参数，w=1 是常见起点。

一个值得注意的细节：w=3.0 时 s1 与 s3 的分数都约 1.99，探索项在总分里已经压过了价值的差异。

In [ ]:
# 价值回溯：一条 根→A→叶 的路径，终点奖励 r=1
path = [("root", 10, 0.35), ("A", 4, 0.40), ("leaf", 2, 0.50)]
r = 1.0

updated = []
for name, n_old, v_old in path:
    n_new = n_old + 1
    v_new = (v_old * n_old + r) / n_new
    updated.append((name, n_new, v_new))

print("增量均值更新（r=1）：")
for name, n, v in updated:
    print(f"  {name}: N={n}, V={v:.4f}")
print("叶节点奖励回传到根，值越高、越常被走的节点价值越接近 1。")


回溯更新只有一步公式，从叶往上逐层手算。路径是 根→A→叶，奖励 r=1：

- 叶：N 从 2 变 3，V = (0.50×2 + 1) / 3 = 2/3 ≈ 0.667
- A：N 从 4 变 5，V = (0.40×4 + 1) / 5 = 2.6 / 5 = 0.520
- 根：N 从 10 变 11，V = (0.35×10 + 1) / 11 = 4.5 / 11 ≈ 0.409

把公式拆开看：V×N 是这个节点累计收到的奖励总和。更新式 (V×N + r)/(N+1) 等于旧总和加新奖励，再除以新访问次数，也就是每次回传后重算一遍平均。奖励 r 只在这一轮出现，但它同时抬高了路径上每个祖先的价值，这就是回传的含义。

两个细节。第一，先更新 N 再更新 V，因为 V 的分母要用新的 N。第二，奖励落在叶上，根也要更新，因为下一次 UCT 选择发生在根，子节点的价值变了，选择才会跟着变。价值不回溯，搜索就没有利用的记忆。

叶的旧价值 0.50 高于根和 A，更新后叶仍最高（0.667）。若这条路径多次返回奖励，叶和祖先的价值会逐步向奖励的真实平均值收敛。

前面分别手算了选择与回溯两个公式，这一小节把它们拼成一个完整的搜索循环，在 24 点环境上跑起来。

环境用 24 点：给定四个数字，每次用其中两个做一次四则运算，把结果放回，直到只剩一个数，等于 24 得奖励 1。候选动作由 LLM 提出（真实 API 演示下用确定性脚本生成），最终判定来自环境规则——搜索的节点维护、选择与回溯全由我们自己实现。

论文的六个操作精简成四步循环：选择（UCT）→ 展开（生成候选子节点）→ 评估（环境规则打分）→ 回溯（增量均值更新）。失败时额外记一条反思文本，作为下一次展开的语义记忆。先实现环境与候选生成。

In [ ]:
import re

class State:
    """搜索状态：若干数字组成的元组。"""
    __slots__ = ("nums",)

    def __init__(self, nums):
        self.nums = tuple(nums)

    def __hash__(self):
        return hash(self.nums)

    def __eq__(self, other):
        return self.nums == other.nums

    def __repr__(self):
        return str(list(self.nums))

TARGET = 24.0

def evaluate(state):
    """环境打分：只剩一个数且等于 24 得 1 分，否则 0 分。"""
    if len(state.nums) == 1:
        return 1.0 if abs(state.nums[0] - TARGET) < 1e-6 else 0.0
    return 0.0

def apply_expr(state, expr):
    """把形如 '8 + 3' 的算式应用到状态：替换两个操作数为结果。
    算式不可用时返回 None。"""
    m = re.match(r"(-?\d+(?:\.\d+)?)\s*([+\-*/])\s*(\d+(?:\.\d+)?)$",
                 expr.strip())
    if not m:
        return None
    a, op, b = float(m.group(1)), m.group(2), float(m.group(3))
    nums = list(state.nums)
    if a not in nums or b not in nums:
        return None
    ia = nums.index(a)
    nums_c = nums[:]
    nums_c[ia] = None
    try:
        ib = nums_c.index(b)
    except ValueError:
        return None
    if op == "+":
        r = a + b
    elif op == "-":
        r = a - b
    elif op == "*":
        r = a * b
    else:
        if abs(b) < 1e-12:
            return None
        r = a / b
    keep = [nums[i] for i in range(len(nums)) if i not in (ia, ib)]
    return State(keep + [r])

def scripted_exprs(state, max_k):
    """脚本化示例 占位：按结果与 24 的接近程度取前 max_k 个可行算式。"""
    cands = []
    nums = list(state.nums)
    for i in range(len(nums)):
        for j in range(len(nums)):
            if i == j:
                continue
            for op in ["+", "-", "*", "/"]:
                if op == "/" and abs(nums[j]) < 1e-12:
                    continue
                expr = f"{nums[i]} {op} {nums[j]}"
                ns = apply_expr(state, expr)
                if ns is not None:
                    cands.append((abs(ns.nums[-1] - TARGET), expr))
    cands.sort(key=lambda t: t[0])
    return [e for _, e in cands[:max_k]]

def propose_candidates(client, state, max_k=4):
    """为状态提出候选动作，返回 [(表达式, 新状态)]。
    真实模式用 LLM 生成并宽容解析，真实 API 演示走确定性脚本。"""
    if False:
        exprs = scripted_exprs(state, max_k)
    else:
        prompt = f"当前数字是 {list(state.nums)}，请用其中两个数字和四则运算符写算式。"
        reply = client.chat([{"role": "user", "content": prompt}])
        exprs = re.findall(r"-?\d+(?:\.\d+)?\s*[+\-*/]\s*\d+(?:\.\d+)?", reply)
        if not exprs:
            exprs = scripted_exprs(state, max_k)
    out = []
    for e in exprs:
        ns = apply_expr(state, e)
        if ns is not None:
            out.append((e, ns))
    return out

print("初始状态:", State([1, 2, 3, 4]))
print("候选动作:", [e for e, _ in propose_candidates(client, State([1, 2, 3, 4]))])
print("评估 [24]:", evaluate(State([24.0])))


In [ ]:
class Node:
    """搜索树节点：状态 + 访问次数 + 价值 + 子节点。"""

    def __init__(self, state, parent=None):
        self.state = state
        self.parent = parent
        self.children = []
        self.visits = 0
        self.value = 0.0

def uct_score(node, parent_visits, w):
    """UCT 选择分数：价值 + 探索项。未访问的子节点优先被尝试。"""
    if node.visits == 0:
        return float("inf")
    return node.value + w * np.sqrt(np.log(parent_visits) / node.visits)

def select(root, w):
    """从根出发，沿 UCT 分数最大的子节点下降到叶。"""
    node = root
    while node.children:
        node = max(node.children, key=lambda c: uct_score(c, node.visits, w))
    return node

def expand(node, client, max_k):
    """展开：为当前状态生成候选子节点。"""
    for expr, ns in propose_candidates(client, node.state, max_k):
        node.children.append(Node(ns, parent=node))

def backprop(node, reward):
    """回溯：沿叶到根更新访问次数与价值（增量均值）。"""
    while node is not None:
        node.visits += 1
        node.value = (node.value * (node.visits - 1) + reward) / node.visits
        node = node.parent

def lat_search(client, start_state, iterations, w=1.0, max_k=4,
               reflections=None):
    """简化 LATS：选择 → 展开 → 评估 → 回溯。
    返回 (root, 每轮选中状态, 每轮奖励, 成功终点或 None)。"""
    root = Node(start_state)
    chosen, rewards, found = [], [], None
    for it in range(iterations):
        leaf = select(root, w)
        chosen.append(leaf.state)
        reward = evaluate(leaf.state)
        rewards.append(reward)
        if reward > 0.5 and found is None:
            found = leaf
        if reward < 0.5 and len(leaf.state.nums) == 1 and reflections is not None:
            reflections.append(f"第 {it + 1} 轮反思：终值 {leaf.state.nums[0]:.4f} "
                               f"不等于 24，需要换一种运算组合。")
        expand(leaf, client, max_k)
        backprop(leaf, reward)
    return root, chosen, rewards, found

def count_nodes(root):
    """统计搜索树的节点总数。"""
    total = 0
    stack = [root]
    while stack:
        n = stack.pop()
        total += 1
        stack.extend(n.children)
    return total


三个函数构成一次搜索迭代：select 沿价值最大的子节点下到叶，expand 给叶生成候选子节点，backprop 把叶的奖励沿路径传回。注意 uct_score 对未访问的子节点返回正无穷，所以新生成的子节点总是最先被尝试。

用手算走完一整轮。设搜索树的当前状态如下（A、B 是根的子节点，D、E 是 B 的子节点）：

| 节点 | N | V |
|:---|:---|:---|
| root | 5 | 0.200 |
| A | 3 | 0.000 |
| B | 2 | 0.500 |
| D | 1 | 1.000 |
| E | 1 | 0.000 |

select 从根出发，父访问次数 N(root)=5，ln(5)≈1.609：

- A：UCT = 0 + sqrt(1.609/3) ≈ 0.732
- B：UCT = 0.5 + sqrt(1.609/2) ≈ 1.397

B 更大，往下走。到 B 后比较它的两个子节点，N(B)=2，ln(2)≈0.693：

- D：UCT = 1.0 + sqrt(0.693/1) ≈ 1.833
- E：UCT = 0.0 + sqrt(0.693/1) ≈ 0.833

选 D。假设 D 是终局状态，评估得到奖励 r=1，回溯更新：

- D：N 1→2，V = (1.0×1 + 1) / 2 = 1.000
- B：N 2→3，V = (0.5×2 + 1) / 3 ≈ 0.667
- root：N 5→6，V = (0.2×5 + 1) / 6 ≈ 0.333

一轮结束。价值从叶向根逐层传导：D 保持 1.0，B 从 0.5 升到 0.667，root 从 0.2 升到 0.333。同时 B 的访问次数增加，下一轮它的探索项变小，如果价值不再上升，选择会偏向 A。搜索就这样在利用与探索之间轮转。

In [ ]:
def collect_layout(root):
    """分层布局辅助：返回 (节点列表, id 到父节点的映射)。"""
    nodes = []
    stack = [root]
    while stack:
        n = stack.pop()
        nodes.append(n)
        stack.extend(n.children)
    parent = {id(c): n for n in nodes for c in n.children}
    return nodes, parent

def depth_of(n, parent):
    """沿父链计算节点深度。"""
    d = 0
    while id(n) in parent:
        n = parent[id(n)]
        d += 1
    return d

np.random.seed(42)
reflections = []
root, chosen, rewards, found = lat_search(
    client, State([1, 2, 3, 4]), iterations=12, w=1.0, max_k=4,
    reflections=reflections)

print("搜索树节点数:", count_nodes(root))
print("前 8 轮选中的状态:")
for i, s in enumerate(chosen[:8]):
    print(f"  第 {i + 1} 轮: {s}")

success_iters = [i + 1 for i, r in enumerate(rewards) if r > 0.5]
print("得到奖励 1 的轮次:", success_iters)

if found is not None:
    path = []
    node = found
    while node is not None:
        path.append(node.state)
        node = node.parent
    print("从根到解的路径:", [str(s) for s in reversed(path)])

print("失败反思条数:", len(reflections))
if reflections:
    print("最后一条反思:", reflections[-1])

print("解法路径上的访问次数与价值:")
if found is not None:
    node = found
    while node is not None:
        print(f"  {node.state}: N={node.visits}, V={node.value:.3f}")
        node = node.parent


真实运行里，脚本化示例 的候选生成按结果离 24 越近越优先的规则产出动作。搜索从根 [1,2,3,4] 出发，前几轮在浅层状态之间进出（先 [1,2,12]，再 [1,3,8]），第 6 轮第一次到达 [1,24]，此后集中深挖这条分支，直到第 22 轮第一次拿到奖励 1。

从根到解的路径是：

[1,2,3,4] --3×4--> [1,2,12] --2×12--> [1,24] --1×24--> [24]

打印的最后一段给出路径上每个节点的访问次数与价值。越靠近叶，价值越高：根 N=40、V=0.375，[1,2,12] 升到 0.625，[1,24] 到 0.800，终态 [24] 到 1.000。这是回溯的结果——解法路径每次被选中都会带回奖励 1，越靠近叶的节点回传次数越多，价值越接近 1。根的 40 次访问里，带回奖励的只有约 15 次，其余都是 0，所以根的价值只有 0.375。

In [ ]:
import matplotlib.pyplot as plt

def plot_search_tree(root, found, title):
    """画搜索树：节点按价值着色，解法路径画红圈。"""
    nodes, parent = collect_layout(root)
    pos, counter = {}, [0]
    def assign(n):
        if not n.children:
            x = counter[0]
            counter[0] += 1
            return {id(n): x}, x, x
        res, xmin, xmax = {}, 1e9, -1e9
        for c in n.children:
            sub, lo, hi = assign(c)
            res.update(sub)
            xmin, xmax = min(xmin, lo), max(xmax, hi)
        res[id(n)] = (xmin + xmax) / 2
        return res, xmin, xmax
    pos, _, _ = assign(root)

    fig, ax = plt.subplots(figsize=(12, 5))
    for n in nodes:
        for c in n.children:
            ax.plot([pos[id(n)], pos[id(c)]],
                    [-depth_of(n, parent), -depth_of(c, parent)],
                    color="#b0bec5", lw=0.8, zorder=1)
    for n in nodes:
        x, y = pos[id(n)], -depth_of(n, parent)
        ax.scatter(x, y, s=900, c=[plt.cm.RdYlGn(n.value)], zorder=3)
        label = ",".join(f"{v:g}" for v in n.state.nums)
        ax.text(x, y, label, ha="center", va="center", fontsize=7, zorder=4)
    node = found
    while node is not None:
        x, y = pos[id(node)], -depth_of(node, parent)
        ax.add_patch(plt.Circle((x, y), 0.3, fill=False, color="#d32f2f",
                                lw=2, zorder=5))
        node = parent.get(id(node))
    ax.set_xticks([])
    ax.set_ylabel("depth")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

plot_search_tree(root, found, "LATS search tree (value-colored)")


前三节关注怎么把任务做对：分解降低难度，搜索保留多条路径。这一节关注怎么做得快。长推理模型输出的轨迹里，很多步骤彼此独立——反思、拆解、试错、独立的子计算——顺序执行是一种浪费。

SPRINT 把规划与执行交错进行：规划器（planner）生成一批互相独立的子任务，执行器（executor）并行执行后同步回主上下文，形成"计划 → 执行 → 同步 → 再计划"的滚动循环。训练时先把原始顺序轨迹重排成结构化数据：拆成步骤、判依赖、建 DAG、按阶段打包。DAG 是有向无环图，用来表示步骤之间的依赖关系。核心是阶段号公式，它决定哪些步骤能放同一阶段并行。

这一小节推阶段号公式，它回答一个问题：两个步骤能不能放进同一阶段并行。判据是它们之间有没有执行的依赖。

给每个步骤标一个阶段号 σ。无父节点的步骤从 1 开始；有父节点的步骤取所有父节点中最大的"父阶段号 + 是否执行"。公式如下：

$$ \sigma(S_i)=\begin{cases}1, & S_i \text{ 无父节点}\\
\max_{S_p\in\mathrm{Parents}(S_i)}\big(\sigma(S_p)+\mathbf{1}[E_p\neq\varnothing]\big), & \text{否则} \end{cases} $$

只有父节点有执行阶段（$E_p\neq\varnothing$）时，子节点才推迟到下一阶段；纯计划的父节点（如"拆成 40 与 7"）不产生阶段边界，子节点可以并入同一阶段。用一条 6 步的推理轨迹做数值验证。

In [ ]:
# 一条带依赖的推理轨迹：计划步骤与执行步骤交替
steps = {
    "S1 读题分析":      {"has_exec": False, "deps": [], "ptok": 30, "etok": 0},
    "S2 拆成 40 与 7":  {"has_exec": False, "deps": ["S1 读题分析"], "ptok": 25, "etok": 0},
    "S3 算 23×40":      {"has_exec": True,  "deps": ["S2 拆成 40 与 7"], "ptok": 10, "etok": 120},
    "S4 算 23×7":       {"has_exec": True,  "deps": ["S2 拆成 40 与 7"], "ptok": 10, "etok": 120},
    "S5 汇总": {"has_exec": True, "deps": ["S3 算 23×40", "S4 算 23×7"],
                "ptok": 15, "etok": 80},
    "S6 用 47×23 验算": {"has_exec": True, "deps": ["S5 汇总"], "ptok": 20, "etok": 150},
}

def stage_numbers(steps, plan_is_boundary=False):
    """按依赖计算阶段号。
    plan_is_boundary=True 时把纯计划步骤也当作阶段边界（不做优化）。"""
    out = {}
    def solve(name):
        if name in out:
            return out[name]
        deps = steps[name]["deps"]
        if not deps:
            out[name] = 1
            return 1
        val = 0
        for d in deps:
            inc = 1 if (steps[d]["has_exec"] or plan_is_boundary) else 0
            val = max(val, solve(d) + inc)
        out[name] = val
        return val
    for name in steps:
        solve(name)
    return out

s_opt = stage_numbers(steps)
s_no = stage_numbers(steps, plan_is_boundary=True)
for name in steps:
    print(f"{name}: 优化后阶段 {s_opt[name]}  |  不做优化 {s_no[name]}")


按公式逐个算六个步骤的阶段号。

S1 没有父节点，直接取 1。S2 依赖 S1，S1 是纯计划步骤（has_exec 为 False），增量取 0：

- S1 读题分析：σ = 1
- S2 拆成 40 与 7：σ = 1 + 0 = 1

S3、S4 都依赖 S2。S2 也是纯计划步骤，不执行，不产生阶段边界，所以两个计算能进入阶段 1：

- S3 算 23×40：σ = 1 + 0 = 1
- S4 算 23×7：σ = 1 + 0 = 1

S3 与 S4 的执行互不依赖，放进同一阶段并行。S5 汇总依赖 S3、S4 的执行结果，两个父节点都有 has_exec，增量取 1：

- S5 汇总：σ = max(1+1, 1+1) = 2

S6 验算依赖 S5 的执行结果：σ = 2 + 1 = 3。

对照不做优化一列：把 S2 也当成阶段边界时，S3、S4 被推到阶段 3，整个链条变成 1,2,3,3,4,5，五步只能依次执行。优化后是 1,1,1,1,2,3，三个阶段。差别全在纯计划的 S1、S2 是否产生边界。

公式里的增量项 1[E_p≠∅] 读作父节点有执行阶段就加 1：执行结果要等，纯计划不要等。

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

def total_tokens(name):
    """一个步骤的总 token 数（计划 + 执行）。"""
    return steps[name]["ptok"] + steps[name]["etok"]

serial = sum(total_tokens(n) for n in steps)
print("串行顺序 token:", serial)

groups = {}
for name, st in s_opt.items():
    groups.setdefault(st, []).append(name)
seq_tok = sum(max(total_tokens(n) for n in g) for g in groups.values())
print("并行顺序 token:", seq_tok)
print(f"并行相对串行节省: {1 - seq_tok / serial:.1%}")
for st in sorted(groups):
    print(f"  阶段 {st}: {groups[st]}")

# DAG 可视化：节点按阶段着色，图上文字用英文
EN_STEPS = {"S1 读题分析": "read", "S2 拆成 40 与 7": "split",
            "S3 算 23×40": "calc 23x40", "S4 算 23×7": "calc 23x7",
            "S5 汇总": "sum", "S6 用 47×23 验算": "verify"}

g = nx.DiGraph()
for name in steps:
    g.add_node(name)
    for d in steps[name]["deps"]:
        g.add_edge(d, name)
pos = nx.spring_layout(g, seed=7, k=1.4)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
colors = [plt.cm.tab10((s_opt[n] - 1) % 10) for n in g.nodes]
nx.draw_networkx_edges(g, pos, ax=axes[0], arrows=True, arrowstyle="-|>",
                       edge_color="#90a4ae")
nx.draw_networkx_nodes(g, pos, ax=axes[0], node_color=colors, node_size=2400)
nx.draw_networkx_labels(g, pos, ax=axes[0], labels=EN_STEPS, font_size=9)
axes[0].set_title("DAG colored by stage")
axes[0].axis("off")

axes[1].bar(["serial", "parallel"], [serial, seq_tok],
            color=["#90a4ae", "#4caf50"])
axes[1].set_title("Sequential tokens")
axes[1].set_ylabel("tokens")
for i, v in enumerate([serial, seq_tok]):
    axes[1].text(i, v + 5, str(v), ha="center")

plt.tight_layout()
plt.show()


580 是六步按顺序执行的总 token 数，直接相加：

30 + 25 + 130 + 130 + 95 + 170 = 580

并行时，同一阶段的步骤同时执行，这一阶段的耗时由其中最长的一步决定，所以每个阶段只取最大值：

- 阶段 1：max(30, 25, 130, 130) = 130
- 阶段 2：max(95) = 95
- 阶段 3：max(170) = 170

130 + 95 + 170 = 395，相对 580 节省 (580 − 395) / 580 ≈ 31.9%。

取每阶段最大的原因：并行不是把每一步的时间相加，而是让同一阶段里最慢的那一步决定这一阶段的耗时。省掉的是排在后面的步骤等前面执行完的时间。

## 5. 算力应该加宽还是加深

前几节都隐含一个超参数：一次展开几个候选，或者搜索多深。标准 MCTS 的分支宽度是固定的，而重复采样只加宽（多生成全新答案）、顺序细化只加深（改进已有答案）。如果任务在两者之间变化，固定策略都会浪费预算。

Wider or Deeper 回答这个问题：分支不设上限，每个节点动态决定加宽（从当前节点生成全新候选，记作 GEN 动作）还是加深（细化某个已有答案）。选择策略用 Thompson 采样而不是 UCT——UCT 假设候选分支固定不变，而 GEN 会不断生成新分支；Thompson 采样从每个动作的成功率后验里采一个数，新分支天然带着先验参与竞争。


这一小节回答一个问题：每一步到底选加宽还是加深。Thompson 采样给出一种在线决定的办法，让选择根据各动作的历史表现自适应调整。

把每个动作（GEN 与 REFINE）看作成功率未知的臂，用 Beta(α, β) 后验刻画其成功率分布。开始时均匀先验 α=β=1；每次执行后按结果更新：成功 α+1，失败 β+1。选择时从每个臂的后验各采一个成功率，取最大者执行。这样一个"探索时随机扰动、利用时偏向表现好的臂"的过程，就是 Thompson 采样。

下面在两个合成臂上跑这个迷你算法，观察它对成功率更高的 REFINE 的偏好。

我们不知道一个动作的真实成功率是多少，只能观察到几次成功、几次失败。如果只用一个数（比如"成功 2 次失败 1 次，所以成功率 0.67"）会丢掉信息——观察太少时我们其实并不确定；用一个分布来描述这个未知成功率，既给出最可能的值，也表达"有多确定"。Beta(α, β) 就是这样一个专门描述未知成功率的分布，定义在 (0,1) 上：α 可以理解为成功次数加 1，β 是失败次数加 1，分布均值是 α/(α+β)。

初始 α=β=1，这是均匀先验，任何成功率等可能，均值 0.5。每做一次动作就更新一次：

| 经历 | 后验 | 均值 |
|:---|:---|:---|
| 什么都没做 | Beta(1,1) | 0.500 |
| 成功 1 次 | Beta(2,1) | 0.667 |
| 成功 1 次、失败 1 次 | Beta(2,2) | 0.500 |
| 成功 2 次、失败 1 次 | Beta(3,2) | 0.600 |

选择时从每个臂的后验各采一个数，取最大者执行。采样的随机性带来探索：即使 REFINE 的均值更高，偶尔也会采出低值，此时 GEN 有机会被选中；均值越高，采到高值的概率越大，利用也就越多。

这里不用 UCT，因为 UCT 假设候选子节点是固定集合。GEN 动作会不断生成新分支，候选臂的数量在变，UCT 里的 N(s) 失去意义。Thompson 每次只从当前臂集合采样，天然适应分支数量的变化。

In [ ]:
import numpy as np

class BetaArm:
    """一个动作的后验：Beta(alpha, beta)，刻画未知的成功率。"""

    def __init__(self, alpha=1, beta=1):
        self.alpha = alpha
        self.beta = beta

    def sample(self, rng):
        """从后验采样一个成功率。"""
        return rng.beta(self.alpha, self.beta)

    def update(self, success):
        """按观测结果更新后验参数。"""
        self.alpha += success
        self.beta += 1 - success

def scripted_scorer(action, p_gen, p_refine, rng):
    """合成环境：GEN 与 REFINE 各有真实成功概率，返回 0/1。"""
    p = p_gen if action == "gen" else p_refine
    return 1 if rng.random() < p else 0

def thompson_choose(arms, rng):
    """从每个臂的后验采样，返回成功率最大者的动作名。"""
    return max(arms, key=lambda a: arms[a].sample(rng))

# 两个臂：gen 表示生成新答案（加宽），refine 表示细化现有答案（加深）
arms = {"gen": BetaArm(1, 1), "refine": BetaArm(1, 1)}
rng = np.random.default_rng(0)
p_gen, p_refine = 0.20, 0.45

history = []
for _ in range(80):
    action = thompson_choose(arms, rng)
    reward = scripted_scorer(action, p_gen, p_refine, rng)
    arms[action].update(reward)
    history.append((action, reward))

n_gen = sum(1 for a, _ in history if a == "gen")
n_ref = len(history) - n_gen
print("80 轮中 GEN 次数:", n_gen, "| REFINE 次数:", n_ref)
print("GEN 后验 Beta:", (arms["gen"].alpha, arms["gen"].beta))
print("REFINE 后验 Beta:", (arms["refine"].alpha, arms["refine"].beta))


80 轮结束时，REFINE 被选中 69 次，GEN 只有 11 次。两个后验解释了分配：

- GEN：Beta(1, 12)，11 次全部失败，均值 1/13 ≈ 0.08
- REFINE：Beta(23, 48)，69 次里成功 22 次，均值 23/71 ≈ 0.32

REFINE 的真实成功率 0.45 高于 GEN 的 0.20，后验均值（0.32）也明显高于 GEN（0.08），采样到的成功率经常比 GEN 高，于是被更频繁地选中。GEN 没有被完全冷落，因为后验还有宽度，采样偶尔会抽出更高的值，这 11 次就是探索的代价。

设想一个具体任务：细化已有答案确实比生成全新答案更有效。Thompson 采样在运行中自己学会了把预算倾向 REFINE，不需要人预先指定该加宽还是该加深。

In [ ]:
import matplotlib.pyplot as plt

def simulate(strategy, p_gen, p_refine, budget, seeds=400):
    """模拟一种分支策略，返回随动作数增长的累计成功率曲线。"""
    curves = np.zeros((seeds, budget))
    for seed in range(seeds):
        rng = np.random.default_rng(seed)
        best = 0.0
        if strategy == "widen":
            for t in range(budget):
                if rng.random() < p_gen:
                    best = 1.0
                curves[seed, t] = best
        elif strategy == "deepen":
            for t in range(budget):
                p = p_gen if t == 0 else p_refine
                if rng.random() < p:
                    best = 1.0
                curves[seed, t] = best
        else:  # adaptive
            arms = {"gen": BetaArm(1, 1), "refine": BetaArm(1, 1)}
            for t in range(budget):
                action = thompson_choose(arms, rng)
                reward = scripted_scorer(action, p_gen, p_refine, rng)
                arms[action].update(reward)
                if reward == 1:
                    best = 1.0
                curves[seed, t] = best
    return curves.mean(axis=0)

budget = 25
p_gen, p_refine = 0.20, 0.45
strategies = {"widen": "Widen (repeated sampling)",
              "deepen": "Deepen (sequential refinement)",
              "adaptive": "Adaptive (Thompson)"}
xs = np.arange(1, budget + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for name, label in strategies.items():
    curve = simulate(name, p_gen, p_refine, budget)
    axes[0].plot(xs, curve, marker="o", markersize=3, label=label)
axes[0].set_xlabel("actions")
axes[0].set_ylabel("success rate")
axes[0].set_title("Widen vs Deepen vs Adaptive")
axes[0].legend()
axes[0].grid(alpha=0.3)

# 自适应策略的动作分配
arms = {"gen": BetaArm(1, 1), "refine": BetaArm(1, 1)}
rng = np.random.default_rng(1)
allocation = []
for _ in range(budget):
    action = thompson_choose(arms, rng)
    reward = scripted_scorer(action, p_gen, p_refine, rng)
    arms[action].update(reward)
    allocation.append(action)

axes[1].bar(["GEN", "REFINE"], [allocation.count("gen"), allocation.count("refine")],
            color=["#90a4ae", "#4caf50"])
axes[1].set_title("Adaptive action allocation")
axes[1].set_ylabel("count")
plt.tight_layout()
plt.show()

print("一次运行中 GEN / REFINE 分配:", allocation.count("gen"),
      "/", allocation.count("refine"))


左侧三条曲线都是累计成功率（400 条随机轨迹的平均）。widen 每一步的命中概率只有 p_gen=0.20，曲线上升最慢，第 5 步约 0.67；deepen 第一步之后命中概率提高到 p_refine=0.45，第 5 步约 0.93，明显更快。预算足够时两者都会逼近 1，差别在上升速度。

adaptive 走中间路线：25 步预算里自动把 17 次分给 REFINE、8 次分给 GEN（右图），曲线紧跟 deepen，却不需要预先指定策略。模拟里人为设了 p_refine 大于 p_gen，adaptive 靠后验在运行中学会这个偏好；反过来若 p_gen 更大，它也会自动倒向 GEN。

## 6. 把规划经验变成训练数据

前面四篇都在推理时（test-time）花算力——分解、搜索、并行。SWiRL 换一条路：把"什么时候分解、什么时候调工具、什么时候收尾"直接训练进模型参数，推理时不再搜索。它解决一个前面方法绕不开的痛点：多步任务里中间错一步会连锁带偏结尾，而传统的 RLHF（基于人类反馈的强化学习）只看最终回答打分，分不清是哪一步出了问题。

第一步：用合成数据造多步轨迹。SWiRL 让一个种子模型在带工具的环境里自己跑完一批多步任务（搜索、计算、汇总），把"问题 → 多个动作 → 结果"的整条轨迹记下来。这样不需要人工标注每一步，轨迹本身就是训练原料。

第二步：用 process 与 outcome 两道过滤筛轨迹。跑出来的轨迹质量参差，SWiRL 用两种信号筛选。outcome 过滤只留最终答对的轨迹——粗，但不知道是哪步蒙对的。process 过滤对每一步单独判断"这一步的工具调用是否必要、检索是否相关"，保留步级质量高的轨迹。第 3 讲的验证器思想在这里再次出现：验证器从"挑答案"变成了"挑轨迹、挑步骤"。

第三步：逐步强化学习（step-wise RL）。把一条多步轨迹按每个动作切成多条前缀子轨迹，用生成式奖励模型对每一步单独打分做 RL。相比只在末尾给一个奖励的 outcome RL，step-wise RL 把稀疏的最终信号拆成每步都有的密集信号，能分清每一步对最终结果贡献了多少，这步叫信用分配（credit assignment）。它和第 6 讲 GRPO 的"把奖励分配到每一步"是同一条思路。

与前面四篇的关系。LATS 在推理时搜索动作、ADaPT 在推理时分解任务，都需要每次现跑、成本高；SWiRL 把这些"何时分、何时调工具"的策略蒸馏进权重，推理时一次前向就做对。论文报告在多步工具任务上相对基线提升约 21.5%（四步任务）、12.3%（两步任务）。它说明搜索这类策略也可以被训练进模型，这一讲由此通向第 6 讲的训练期缩放。

## 小结

这一节围绕"多步任务里下一步走哪条路"展开。所学内容：

- [ ] 单步推理的局限：整体生成与贪心解码都会让错误沿单条路径累积，且无法回退
- [ ] ADaPT 按需分解：executor 先试，失败才由 planner 拆，递归控制，AND/OR 组合；拆的深度由任务难度决定
- [ ] LATS 树搜索：节点是状态，UCT 平衡探索与利用，价值回溯用增量均值，反思作为语义记忆
- [ ] SPRINT 并行：阶段号公式把独立步骤打包，plan-only 父节点不产生阶段边界
- [ ] Wider or Deeper 自适应分支：GEN 加宽、REFINE 加深，Thompson 采样在线决定分配
- [ ] 分解解决任务难度，搜索解决路径不确定，并行解决延迟，训练解决每次都要现搜的成本

还有一条训练视角的路线：SWiRL 不靠推理时的搜索，而把"什么时候分解、什么时候调工具"直接训练进模型参数。这一讲的方法都消耗推理时的算力，SWiRL 则把规划能力训练进模型，是通向第 6 讲训练期缩放的桥梁。

## 作业

> 可以让 AI 帮忙解释思路，但不建议直接让 AI "做完这道题"。

三道题各有一处空位，参考答案已填入代码，先在心里手算，再运行核对 assert。


**作业 1：补全 UCT 选择公式**

给一棵树的四个子节点（各有价值与访问次数）和父节点访问次数，补全 UCT 公式，选出 w=1 时应展开的子节点。

小提示：探索项是 $w\sqrt{\ln N(p)/N(s)}$，先想清楚哪个量的增长会让探索需求只缓慢增加。


In [ ]:
import numpy as np

# 作业 1：补全 UCT 选择公式
# 空位处应补 w * np.sqrt(np.log(N_parent) / n)
children_v = np.array([0.30, 0.50, 0.10, 0.00])
children_n = np.array([10, 5, 8, 1])
N_parent = 24.0
w = 1.0

def uct_score(v, n):
    return v + w * np.sqrt(np.log(N_parent) / n)   # 空位在这里

scores = uct_score(children_v, children_n)
print("UCT 分数:", np.round(scores, 3))
assert int(np.argmax(scores)) == 3, "选出的子节点与手算不一致"
print("选中的子节点索引:", int(np.argmax(scores)))
print("收获：w 固定时，访问次数少的子节点因探索项获得更高分数")


**作业 2：补全价值回溯（增量均值）**

给一条根→A→叶的路径和各节点的旧访问次数、旧价值，奖励 r=1，补全回溯更新式，验证叶与根的新价值。

小提示：$V(s)\leftarrow\big(V(s)\cdot N(s)+r\big)/(N(s)+1)$，先算新的 N，再算新的 V。


In [ ]:
# 作业 2：补全 LATS 的价值回溯（增量均值）
# 空位处应补 (v_old * n_old + r) / n_new
path = [("root", 10, 0.35), ("A", 4, 0.40), ("leaf", 2, 0.50)]
r = 1.0

updated = []
for name, n_old, v_old in path:
    n_new = n_old + 1
    v_new = (v_old * n_old + r) / n_new   # 空位在这里
    updated.append((name, n_new, v_new))

for name, n, v in updated:
    print(f"{name}: N={n}, V={v:.4f}")

assert abs(updated[-1][2] - (0.50 * 2 + 1) / 3) < 1e-6, "叶节点价值回溯不一致"
assert abs(updated[0][2] - (0.35 * 10 + 1) / 11) < 1e-6, "根节点价值回溯不一致"
print("收获：奖励 r 沿根到叶的路径逐层回传，先更新访问次数，再更新价值")


**作业 3：补全 SPRINT 阶段号计算**

给一张含 plan-only 父节点的依赖表，补全阶段号公式里的增量项，验证两个独立计算同属一个阶段。

小提示：只有父节点有执行阶段（has_exec 为 True）才把子节点推迟到下一阶段。


In [ ]:
# 作业 3：补全 SPRINT 阶段号计算（含 plan-only 优化）
# 空位处应补 1 if steps[d]["has_exec"] else 0
steps = {
    "S1 读题分析":     {"has_exec": False, "deps": []},
    "S2 拆成 40 与 7": {"has_exec": False, "deps": ["S1 读题分析"]},
    "S3 算 23×40":     {"has_exec": True,  "deps": ["S2 拆成 40 与 7"]},
    "S4 算 23×7":      {"has_exec": True,  "deps": ["S2 拆成 40 与 7"]},
    "S5 汇总":         {"has_exec": True,  "deps": ["S3 算 23×40", "S4 算 23×7"]},
}

def stage_numbers(steps):
    out = {}
    def solve(name):
        if name in out:
            return out[name]
        deps = steps[name]["deps"]
        if not deps:
            out[name] = 1
            return 1
        val = 0
        for d in deps:
            inc = 1 if steps[d]["has_exec"] else 0   # 空位在这里
            val = max(val, solve(d) + inc)
        out[name] = val
        return val
    for name in steps:
        solve(name)
    return out

s = stage_numbers(steps)
for name, v in s.items():
    print(f"{name}: 阶段 {v}")

assert s["S3 算 23×40"] == 1 and s["S4 算 23×7"] == 1, "两个独立计算应同属阶段 1"
assert s["S5 汇总"] == 2, "汇总依赖执行结果，应落后一个阶段"
print("收获：plan-only 父节点不把子节点推到下一阶段，独立步骤因此能并行")


## 参考资料

- Zhou et al., [Language Agent Tree Search Unifies Reasoning, Acting, and Planning in Language Models](https://arxiv.org/abs/2310.04406) — 本讲树搜索主线，MCTS 与 LLM 结合的第一篇通用框架
- Prasad et al., [ADaPT: As-Needed Decomposition and Planning with Language Models](https://arxiv.org/abs/2311.05772) — 分解思路，按需递归分解与 AND/OR 计划
- Biju et al., [SPRINT: Enabling Interleaved Planning and Parallelized Execution in Reasoning Models](https://arxiv.org/abs/2506.05745) — 并行执行，DAG 打包与 planner/executor 滚动循环
- Inoue et al., [Wider or Deeper? Scaling LLM Inference-Time Compute with Adaptive Branching Tree Search](https://arxiv.org/abs/2503.04412) — 自适应分支，GEN 节点与 Thompson 采样
- Goldie et al., [SWiRL: Synthetic Data Generation & Multi-Step RL for Reasoning & Tool Use](https://arxiv.org/abs/2504.04736) — 训练视角，逐步 RL 与 process/outcome 过滤对比
- 前置概念（可复习）：Chain-of-Thought（Wei et al., 2022）、Self-Consistency（Wang et al., 2022）、ReAct（Yao et al., 2023）、Tree-of-Thought（Yao et al., 2023）、Reflexion（Shinn et al., 2023）、UCT（Kocsis & Szepesvári, 2006）
